## 00. load package

In [1]:

import intake
import cartopy.crs as ccrs
import cartopy.feature as cf
import cmocean
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import easygems.healpix as egh
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from typing import Tuple, List, Optional, Union, Dict
import scipy.signal as signal
from global_land_mask import globe
import math
import time
import os
import pickle
from scipy import fft
import cmaps  
from typing import Optional
import sys
import matplotlib.colors as mcolors
from pathlib import Path
WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))
from wave_tools.utils import dataarray_to_equatorial_latlon_grid,get_region_healpix_,create_cmap_from_string,dataarray_healpix_to_equatorial_latlon
from  wave_tools.plotting import get_cckw_envelope_curve,setup_map_axes,set_axis_for_wave    
# 重新导入修复后的模块
import wave_tools.spectral
import wave_tools.plotting
from wave_tools.spectral import WKSpectralAnalysis, SpectralConfig
from wave_tools.plotting import set_axis_for_wave
from wave_tools import matsuno as mp
print("="*70)
print("✅ 模块重新加载完成")
print("="*70)
import mpi4py
import logging
import glob

# ============= 新增：Dask 配置优化 =============
import dask
from dask.diagnostics import ProgressBar

# 配置 Dask 以避免内存溢出
dask.config.set({
    'array.slicing.split_large_chunks': True,
    'distributed.worker.memory.target': 0.75,  # 75% 内存使用上限
    'distributed.worker.memory.spill': 0.85,   # 85% 时开始写入磁盘
    'distributed.worker.memory.pause': 0.95,   # 95% 时暂停
    'array.chunk-size': '128MiB'  # 设置合理的分块大小
})

print("✅ Dask 内存优化配置完成")
print("="*70)


✅ 模块重新加载完成
✅ Dask 内存优化配置完成


In [2]:
# 网格转换参数

def dataarray_to_equatorial_latlon_grid(
    dataarray: xr.DataArray, grid_type: str, grid_dict: Optional[dict]
) -> xr.DataArray:
    """转换数据到赤道经纬度网格"""
    if grid_type == "latlon":
        return dataarray
    elif grid_type == "healpix":
        if grid_dict is None:
            raise ValueError("No grid_dict provided for healpix conversion.")
        return dataarray_healpix_to_equatorial_latlon(dataarray, **grid_dict)
    else:
        raise ValueError("Grid type not found.")




In [3]:
cat = intake.open_catalog("https://data.nextgems-h2020.eu/catalog.yaml")
# 创建数据保存目录
DATA_DIR = "/work/mh1498/m301257/processed_data"
os.makedirs(DATA_DIR, exist_ok=True)
ds = (cat.ICON.C5.AMIP_CNTL.to_dask()).sel(time=slice("1980", "1993"))
ds 


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


<xarray.Dataset> Size: 6TB
Dimensions:             (time: 5114, cell: 786432, level_full: 26,
                         level_half: 26)
Coordinates:
  * time                (time) datetime64[ns] 41kB 1980-01-01 ... 1993-12-31
  * level_full          (level_full) float64 208B 14.0 21.0 25.0 ... 89.0 90.0
  * level_half          (level_half) float64 208B 14.0 21.0 25.0 ... 89.0 90.0
    healpix             int64 8B 1
Dimensions without coordinates: cell
Data variables: (12/47)
    clivi               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    cllvi               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hus2m               (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hfls                (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    hfss                (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    pr                  (time, cell) float32 16GB dask.array<chunksize=(128, 16384), meta=np.ndarray>
    ...                  ...
    phalf               (time, level_half, cell) float32 418GB dask.array<chunksize=(32, 4, 16384), meta=np.ndarray>
    cell_elevation      (cell) float64 6MB dask.array<chunksize=(262144,), meta=np.ndarray>
    cell_sea_land_mask  (cell) int32 3MB dask.array<chunksize=(262144,), meta=np.ndarray>
    zg                  (level_full, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
    zghalf              (level_half, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
    dzghalf             (level_full, cell) float32 82MB dask.array<chunksize=(1, 262144), meta=np.ndarray>
Attributes:
    CDI:                       Climate Data Interface version 2.4.0 (https://...
    Conventions:               CF-1.6
    source:                    https://gitlab.dkrz.de/icon/icon-mpim.git@6684...
    institution:               Max Planck Institute for Meteorology/Deutscher...
    title:                     ICON simulation
    references:                see MPIM/DWD publications
    comment:                   Lukas Kluft (kluftluka) on nid006406 (Linux 5....
    cdo_bitrounding_numbits:   13
    CDO:                       Climate Data Operators version 2.4.0 (https://...
    cdo_openmp_thread_number:  4
    history:                   Wed Mar 13 20:19:59 2024: ncrename -d cells,ce...
    NCO:                       netCDF Operators version 5.0.1 (Homepage = htt...

In [4]:
# 检查系统资源
import psutil

def check_system_resources():
    """检查并打印系统资源使用情况"""
    # 内存
    mem = psutil.virtual_memory()
    print("="*70)
    print("💻 系统资源状态")
    print("="*70)
    print(f"内存总量: {mem.total / (1024**3):.2f} GB")
    print(f"内存可用: {mem.available / (1024**3):.2f} GB")
    print(f"内存使用率: {mem.percent}%")
    
    # CPU
    cpu_percent = psutil.cpu_percent(interval=1)
    print(f"CPU使用率: {cpu_percent}%")
    print(f"CPU核心数: {psutil.cpu_count()}")
    
    # 磁盘
    disk = psutil.disk_usage('/work')
    print(f"磁盘总量: {disk.total / (1024**3):.2f} GB")
    print(f"磁盘可用: {disk.free / (1024**3):.2f} GB")
    print(f"磁盘使用率: {disk.percent}%")
    print("="*70)
    
    # 警告检查
    if mem.percent > 85:
        print("⚠️ 警告: 内存使用率过高！建议减小批次大小或重启内核")
    if disk.percent > 90:
        print("⚠️ 警告: 磁盘空间不足！")
    
    return mem.available / (1024**3)  # 返回可用内存(GB)

available_memory = check_system_resources()


💻 系统资源状态
内存总量: 501.87 GB
内存可用: 274.21 GB
内存使用率: 45.4%
CPU使用率: 0.8%
CPU核心数: 256
磁盘总量: 123638433.80 GB
磁盘可用: 24635452.11 GB
磁盘使用率: 79.8%


In [5]:
def process_var_data(var_name, experiment_name, dataset_key, save_dir, grid_dict, target_lat, target_lon, 
                     has_level=True, level_slice=(30, None)):
    """
    处理变量数据（支持3D和2D），转换网格并插值后保存
    
    Parameters:
    -----------
    var_name : str
        变量名 ('wa', 'hus', 'ta', 'pr', 'phalf', 'pfull')
    experiment_name : str
        实验名称（用于显示）
    dataset_key : str
        在catalog中的数据集键名
    save_dir : str
        保存目录
    grid_dict : dict
        网格转换参数
    target_lat : array
        目标纬度
    target_lon : array
        目标经度
    has_level : bool
        是否为3D数据（有level维度）。True=3D，False=2D
    level_slice : tuple
        level切片范围，仅当has_level=True时有效
    """
    import time
    import gc  # 垃圾回收
    
    data_type = "3D (多层级)" if has_level else "2D (时间序列)"
    print("="*70)
    print(f"🔄 处理 {var_name.upper()} - {experiment_name} [{data_type}]")
    print("="*70)
    
    # 创建变量和实验的子目录
    if has_level:
        exp_save_dir = os.path.join(save_dir, f"{var_name}_{experiment_name.lower()}_layers")
    else:
        exp_save_dir = os.path.join(save_dir, f"{var_name}_{experiment_name.lower()}")
    os.makedirs(exp_save_dir, exist_ok=True)
    print(f"📁 保存路径: {exp_save_dir}")
    
    # 加载数据（只读取元数据）
    print(f"📖 读取数据元信息...")
    
    # 自动识别 level 维度名称
    level_dim_name = None
    
    try:
        if has_level:
            # 先加载变量以检查其维度
            var_temp = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(time=slice("1980", "1993"))
            
            # 自动检测 level 维度名称
            possible_level_dims = ['level_full', 'level_half', 'level']
            for dim in possible_level_dims:
                if dim in var_temp.dims:
                    level_dim_name = dim
                    break
            
            if level_dim_name is None:
                raise ValueError(f"无法找到 level 维度。变量 {var_name} 的维度: {list(var_temp.dims)}")
            
            print(f"✅ 自动检测到 level 维度: {level_dim_name}")
            
            # 根据检测到的维度名称选择数据
            var_full = var_temp.sel({level_dim_name: slice(*level_slice)})
            levels = var_full[level_dim_name].values
            n_levels = len(levels)
            
            print(f"✅ 数据信息:")
            print(f"   变量: {var_name}")
            print(f"   时间范围: 1980-1993")
            print(f"   总层数: {n_levels}")
            print(f"   层级范围: {levels[0]:.1f} - {levels[-1]:.1f}")
            print(f"   时间步数: {len(var_full.time)}")
        else:
            # 2D数据：只有时间维度
            var_full = cat.ICON.C5[dataset_key].to_dask()[var_name].sel(
                time=slice("1980", "1993")
            )
            
            print(f"✅ 数据信息:")
            print(f"   变量: {var_name}")
            print(f"   时间范围: 1980-1993")
            print(f"   时间步数: {len(var_full.time)}")
    except Exception as e:
        print(f"❌ 数据加载失败: {str(e)}")
        return
    
    print("="*70)
    
    # 逐层处理（3D）或整体处理（2D）
    total_start_time = time.time()
    
    if has_level:
        # ========== 3D数据：逐层处理 ==========
        for idx, level in enumerate(levels, 1):
            layer_start_time = time.time()
            
            # 构建保存路径
            save_path = os.path.join(exp_save_dir, f"{var_name}_lev_{int(level):03d}.nc")
            
            # 检查是否已处理
            if os.path.exists(save_path):
                print(f"✅ [{idx}/{n_levels}] Level {int(level):3d} - 已存在，跳过")
                continue
            
            print(f"🔄 [{idx}/{n_levels}] 处理 Level {int(level):3d}...")
            
            try:
                # 1. 选择单层数据（使用自动检测的维度名称）
                var_layer = var_full.sel({level_dim_name: level})
                print(f"   ├─ 选择层级完成 (使用 {level_dim_name})")
                
                # 2. 转换到经纬度网格
                var_lonlat = dataarray_to_equatorial_latlon_grid(var_layer, 'healpix', grid_dict)
                print(f"   ├─ 网格转换完成: {var_lonlat.shape}")
                
                # 3. 插值到2°x2°
                var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
                print(f"   ├─ 插值完成: {var_2deg.shape}")
                
                # 4. 转换为Dataset并设置变量名，然后保存
                ds_to_save = var_2deg.to_dataset(name=var_name)
                
                # 使用 compute() 触发计算并保存
                with ProgressBar():
                    ds_to_save.to_netcdf(save_path, compute=True)
                
                # 清理内存
                del var_layer, var_lonlat, var_2deg, ds_to_save
                gc.collect()
                
                # 计算时间
                layer_time = time.time() - layer_start_time
                elapsed_time = time.time() - total_start_time
                avg_time_per_layer = elapsed_time / idx
                remaining_layers = n_levels - idx
                estimated_remaining = avg_time_per_layer * remaining_layers
                
                print(f"   ✅ 保存完成: {os.path.basename(save_path)}")
                print(f"   ⏱️  本层耗时: {layer_time:.1f}s | 平均: {avg_time_per_layer:.1f}s/层")
                print(f"   📊 预计剩余时间: {estimated_remaining/60:.1f} 分钟")
                print()
                
            except MemoryError:
                print(f"   ❌ 内存不足，跳过此层")
                gc.collect()
                continue
            except Exception as e:
                print(f"   ❌ 处理失败: {str(e)}")
                gc.collect()
                continue
    
    else:
        # ========== 2D数据：分批处理避免内存溢出 ==========
        save_path = os.path.join(exp_save_dir, f"{var_name}_2deg_interp.nc")
        
        # 检查是否已处理
        if os.path.exists(save_path):
            print(f"✅ 数据已存在，跳过处理")
            print(f"   文件: {save_path}")
        else:
            print(f"🔄 开始处理2D数据...")
            
            try:
                # 分批处理时间步以避免内存溢出
                n_times = len(var_full.time)
                batch_size = 365 * 2  # 每次处理2年数据
                n_batches = int(np.ceil(n_times / batch_size))
                
                print(f"   ├─ 总时间步: {n_times}")
                print(f"   ├─ 批次大小: {batch_size}")
                print(f"   ├─ 总批次数: {n_batches}")
                
                processed_data = []
                
                for batch_idx in range(n_batches):
                    start_idx = batch_idx * batch_size
                    end_idx = min((batch_idx + 1) * batch_size, n_times)
                    
                    print(f"   ├─ 批次 {batch_idx+1}/{n_batches}: 处理时间步 {start_idx}-{end_idx}...")
                    
                    # 选择批次数据
                    var_batch = var_full.isel(time=slice(start_idx, end_idx))
                    
                    # 转换到经纬度网格
                    var_lonlat = dataarray_to_equatorial_latlon_grid(var_batch, 'healpix', grid_dict)
                    
                    # 插值到2°x2°
                    var_2deg = var_lonlat.interp(lat=target_lat, lon=target_lon, method='linear')
                    
                    # 立即计算并存储结果
                    with ProgressBar():
                        var_2deg_computed = var_2deg.compute()
                    
                    processed_data.append(var_2deg_computed)
                    
                    # 清理内存
                    del var_batch, var_lonlat, var_2deg
                    gc.collect()
                    
                    print(f"   ├─ 批次 {batch_idx+1}/{n_batches} 完成")
                
                # 合并所有批次
                print(f"   ├─ 合并所有批次...")
                var_final = xr.concat(processed_data, dim='time')
                
                # 保存
                print(f"   ├─ 保存中...")
                ds_to_save = var_final.to_dataset(name=var_name)
                ds_to_save.to_netcdf(save_path)
                
                # 清理内存
                del var_final, ds_to_save, processed_data
                gc.collect()
                
                total_time = time.time() - total_start_time
                print(f"   ✅ 保存完成: {os.path.basename(save_path)}")
                print(f"   ⏱️  总耗时: {total_time/60:.1f} 分钟")
                
            except MemoryError:
                print(f"   ❌ 内存不足，尝试减小批次大小")
                gc.collect()
                return
            except Exception as e:
                print(f"   ❌ 处理失败: {str(e)}")
                gc.collect()
                return
    
    total_time = time.time() - total_start_time
    print("="*70)
    print(f"✅ {var_name.upper()} - {experiment_name} 处理完成!")
    print(f"   总耗时: {total_time/60:.1f} 分钟")
    print(f"   保存目录: {exp_save_dir}")
    print("="*70)
    print()
    
    # 最后清理
    gc.collect()


## 循环转换数据-气压层

In [6]:


# 网格参数
grid_dict = {"nside": 256, "nest": True, "minmax_lat": 16}
target_lat = np.arange(-16, 16.1, 2.0)
target_lon = np.arange(0, 360, 2.0)

# 要处理的变量列表（3D和2D）
variables_3d = [
    # 'phalf'
    'rho'
    # "ua", "va"
    # "pfull"
    
    ]  # 3D变量：需要逐层处理
variables_2d = [
      
            # "hfls", "hfss", 
            #     "rsdt", "rsut", "rlut",   
            #     "rsds", "rsus", "rlds", "rlus",
                # "sfcwind",
                # "ts","tas"
                # "hus2m"
                # "ps"
                # "zg"
                ]
          
if variables_2d:
    # 设置保存目录
    LAYER_DIR = os.path.join(DATA_DIR, "2d_layers")
    os.makedirs(LAYER_DIR, exist_ok=True)
else:
    LAYER_DIR = os.path.join(DATA_DIR, "3d_layers")
    os.makedirs(LAYER_DIR, exist_ok=True)   
print("="*70)
print("🚀 开始处理所有变量")
print("="*70)
# print(f"3D变量: {len(variables_3d)} ({', '.join(variables_3d)})")
print(f"2D变量: {len(variables_2d)} ({', '.join(variables_2d)})")
print(f"实验数量: 3 (CNTL, P4K, 4CO2)")
print(f"目标分辨率: 2° x 2°")
print(f"纬度范围: -14° to 14°")
print("="*70)

# 检查资源
check_system_resources()
print()

# 图片保存路径
fig_save_path = os.path.join("/home/m/m301257/", "fig") 
os.makedirs(fig_save_path, exist_ok=True)

print("="*70)
print("📁 数据保存路径设置完成")
print("="*70)
print(f"3D数据目录: {LAYER_DIR}")
print(f"图片目录: {fig_save_path}")
print("="*70)
print()

# 定义实验配置
experiments = {
    "cntl":  ("CNTL",   "AMIP_CNTL"),
    "4k":    ("P4K",    "AMIP_P4K"),
    "4co2":  ("4CO2",   "AMIP_4CO2"),
}

# 处理所有变量和实验
all_start_time = time.time()

# 记录失败的任务
failed_tasks = []

# # 处理3D变量
for var_name in variables_3d:
    print("\n" + "="*70)
    print(f"📊 开始处理3D变量: {var_name.upper()}")
    print("="*70 + "\n")
    
    for exp_key, (exp_name, dataset_key) in experiments.items():
        try:
            # 处理前检查内存
            mem = psutil.virtual_memory()
            if mem.percent > 90:
                print(f"⚠️ 内存使用率过高 ({mem.percent}%)，跳过此任务，等待内存释放")
                time.sleep(30)  # 等待30秒
                gc.collect()
                continue
            
            process_var_data(
                var_name=var_name,
                experiment_name=exp_name,
                dataset_key=dataset_key,
                save_dir=LAYER_DIR,
                grid_dict=grid_dict,
                target_lat=target_lat,
                target_lon=target_lon,
                has_level=True  # 3D数据
            )
        except MemoryError as e:
            error_msg = f"{var_name.upper()} - {exp_name}: 内存不足"
            print(f"❌ {error_msg}")
            failed_tasks.append(error_msg)
            gc.collect()
            time.sleep(10)
            continue
        except Exception as e:
            error_msg = f"{var_name.upper()} - {exp_name}: {str(e)}"
            print(f"❌ {error_msg}")
            failed_tasks.append(error_msg)
            gc.collect()
            continue

# 处理2D变量
for var_name in variables_2d:
    print("\n" + "="*70)
    print(f"📊 开始处理2D变量: {var_name.upper()}")
    print("="*70 + "\n")
    
    for exp_key, (exp_name, dataset_key) in experiments.items():
        try:
            # 处理前检查内存
            mem = psutil.virtual_memory()
            if mem.percent > 90:
                print(f"⚠️ 内存使用率过高 ({mem.percent}%)，跳过此任务，等待内存释放")
                time.sleep(30)  # 等待30秒
                gc.collect()
                continue
            
            process_var_data(
                var_name=var_name,
                experiment_name=exp_name,
                dataset_key=dataset_key,
                save_dir=LAYER_DIR,
                grid_dict=grid_dict,
                target_lat=target_lat,
                target_lon=target_lon,
                has_level=False  # 2D数据
            )
        except MemoryError as e:
            error_msg = f"{var_name.upper()} - {exp_name}: 内存不足"
            print(f"❌ {error_msg}")
            failed_tasks.append(error_msg)
            gc.collect()
            time.sleep(10)
            continue
        except Exception as e:
            error_msg = f"{var_name.upper()} - {exp_name}: {str(e)}"
            print(f"❌ {error_msg}")
            failed_tasks.append(error_msg)
            gc.collect()
            continue

all_total_time = time.time() - all_start_time
print("\n" + "="*70)
print("🎉 所有变量和实验处理完成!")
# print(f"3D变量数: {len(variables_3d)}")
print(f"2D变量数: {len(variables_2d)}")
print(f"实验数: {len(experiments)}")
print(f"总耗时: {all_total_time/60:.1f} 分钟 ({all_total_time/3600:.2f} 小时)")

# 显示失败任务
if failed_tasks:
    print("\n" + "="*70)
    print("⚠️ 以下任务处理失败:")
    print("="*70)
    for task in failed_tasks:
        print(f"  - {task}")
else:
    print("\n✅ 所有任务成功完成！")

print("="*70)

# 最终资源检查
print("\n")
check_system_resources()


🚀 开始处理所有变量
2D变量: 0 ()
实验数量: 3 (CNTL, P4K, 4CO2)
目标分辨率: 2° x 2°
纬度范围: -14° to 14°
💻 系统资源状态
内存总量: 501.87 GB
内存可用: 274.16 GB
内存使用率: 45.4%
CPU使用率: 1.1%
CPU核心数: 256
磁盘总量: 123638433.80 GB
磁盘可用: 24635452.11 GB
磁盘使用率: 79.8%

📁 数据保存路径设置完成
3D数据目录: /work/mh1498/m301257/processed_data/3d_layers
图片目录: /home/m/m301257/fig


📊 开始处理3D变量: RHO

🔄 处理 RHO - CNTL [3D (多层级)]
📁 保存路径: /work/mh1498/m301257/processed_data/3d_layers/rho_cntl_layers
📖 读取数据元信息...
✅ 自动检测到 level 维度: level_half
✅ 数据信息:
   变量: rho
   时间范围: 1980-1993
   总层数: 22
   层级范围: 31.0 - 90.0
   时间步数: 5114
🔄 [1/22] 处理 Level  31...
   ├─ 选择层级完成 (使用 level_half)
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 211
DEBUG: Reference latitude: -15.87, with 1024 longitude points
  Lat -15.8689: 1024 points, target: 1024 points
  Lat -15.7139: 1024 points, target: 1024 points
  Lat -15.5589: 1024 points, target: 1024 points
  Lat 15.8689: 1024 points, target: 1024 points
   ├

/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


✅ 自动检测到 level 维度: level_half
✅ 数据信息:
   变量: rho
   时间范围: 1980-1993
   总层数: 22
   层级范围: 31.0 - 90.0
   时间步数: 5114
🔄 [1/22] 处理 Level  31...
   ├─ 选择层级完成 (使用 level_half)
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 211
DEBUG: Reference latitude: -15.87, with 1024 longitude points
  Lat -15.8689: 1024 points, target: 1024 points
  Lat -15.7139: 1024 points, target: 1024 points
  Lat -15.5589: 1024 points, target: 1024 points
  Lat 15.8689: 1024 points, target: 1024 points
   ├─ 网格转换完成: (5114, 211, 1024)
   ├─ 插值完成: (5114, 17, 180)
   ✅ 保存完成: rho_lev_031.nc
   ⏱️  本层耗时: 348.2s | 平均: 348.2s/层
   📊 预计剩余时间: 121.9 分钟

🔄 [2/22] 处理 Level  35...
   ├─ 选择层级完成 (使用 level_half)
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 211
DEBUG: Reference latitude: -15.87, with 1024 longitude points
  Lat -15.8689: 1024 points, target: 1024 points
 

/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


✅ 自动检测到 level 维度: level_half
✅ 数据信息:
   变量: rho
   时间范围: 1980-1993
   总层数: 22
   层级范围: 31.0 - 90.0
   时间步数: 5114
🔄 [1/22] 处理 Level  31...
   ├─ 选择层级完成 (使用 level_half)
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 211
DEBUG: Reference latitude: -15.87, with 1024 longitude points
  Lat -15.8689: 1024 points, target: 1024 points
  Lat -15.7139: 1024 points, target: 1024 points
  Lat -15.5589: 1024 points, target: 1024 points
  Lat 15.8689: 1024 points, target: 1024 points
   ├─ 网格转换完成: (5114, 211, 1024)
   ├─ 插值完成: (5114, 17, 180)
   ✅ 保存完成: rho_lev_031.nc
   ⏱️  本层耗时: 332.1s | 平均: 332.1s/层
   📊 预计剩余时间: 116.2 分钟

🔄 [2/22] 处理 Level  35...
   ├─ 选择层级完成 (使用 level_half)
DEBUG: Total pixels: 786432
DEBUG: Lat range: -89.82 to 89.82
DEBUG: Lon range: 0.00 to 359.82
DEBUG: Unique latitudes in range: 211
DEBUG: Reference latitude: -15.87, with 1024 longitude points
  Lat -15.8689: 1024 points, target: 1024 points
 

267.4857406616211

## 合并变量，修改变量名称，合并数据

In [7]:
def change_variable_name_and_merge_per_folder(in_paths, base_out_path, pattern,
                                              old_var_name, new_var_name,
                                              merged_file_name="wa_all_levels.nc",
                                              skip_existing=True,
                                              has_level=True):
    """
    批量更改NetCDF文件中的变量名，添加level维度（如果需要），并在每个子文件夹生成合并文件
    同时保留输入子文件夹结构
    
    Parameters:
    -----------
    skip_existing : bool
        如果为True，跳过已存在的单个文件和合并文件（默认True）
    has_level : bool
        如果为True，处理3D数据（从文件名提取level维度）
        如果为False，处理2D数据（直接合并，不添加level维度）
    """
    import os
    import glob
    import xarray as xr

    for in_path in in_paths:
        # 当前输入文件夹名称
        folder_name = os.path.basename(os.path.normpath(in_path))
        out_path = os.path.join(base_out_path, folder_name)
        os.makedirs(out_path, exist_ok=True)
        
        # 检查合并文件是否已存在
        merged_file = os.path.join(out_path, merged_file_name)
        if skip_existing and os.path.exists(merged_file):
            print(f"✅ {folder_name}: 合并文件已存在，跳过处理")
            print(f"   文件: {merged_file}")
            continue

        file_pattern = os.path.join(in_path, pattern)
        input_files = sorted(glob.glob(file_pattern))
        
        if not input_files:
            print(f"⚠️ 文件夹 {folder_name} 没有匹配的文件 (pattern: {pattern})")
            continue
        
        data_type = "3D (多层级)" if has_level else "2D (单层/时间序列)"
        print(f"\n{'='*70}")
        print(f"🔄 处理文件夹: {folder_name} [{data_type}]")
        print(f"   输入路径: {in_path}")
        print(f"   输出路径: {out_path}")
        print(f"   找到文件数: {len(input_files)}")
        print(f"{'='*70}")
        
        datasets = []
        processed_count = 0
        skipped_count = 0

        for file in input_files:
            filename = os.path.basename(file)
            new_file = os.path.join(out_path, filename)

            # 检查单个文件是否已存在
            if skip_existing and os.path.exists(new_file):
                skipped_count += 1
                # 仍需加载用于合并
                try:
                    with xr.open_dataset(new_file) as ds:
                        if has_level:
                            # 3D数据：需要level维度
                            level = int(filename.split("_")[2].replace(".nc", ""))
                            ds = ds.expand_dims({"level": [level]}) if "level" not in ds.dims else ds
                        datasets.append(ds)
                except Exception as e:
                    print(f"⚠️ 跳过的文件加载失败: {filename} - {str(e)}")
                continue

            # 处理新文件
            try:
                if has_level:
                    # 3D数据：从文件名提取 level (例如: hus_lev_031.nc -> 31)
                    try:
                        level = int(filename.split("_")[2].replace(".nc", ""))
                    except:
                        print(f"⚠️ 无法从文件名提取 level: {filename}, 跳过")
                        continue
                
                with xr.open_dataset(file) as ds:
                    # 修改变量名（如果需要）
                    if old_var_name in ds and old_var_name != new_var_name:
                        ds = ds.rename({old_var_name: new_var_name})
                    
                    # 3D数据需要添加 level 维度
                    if has_level:
                        ds = ds.expand_dims({"level": [level]})
                    
                    # 保存到新路径
                    ds.to_netcdf(new_file, mode="w")
                    processed_count += 1
                    
                    if processed_count % 5 == 0 or processed_count == len(input_files):
                        print(f"✅ [{processed_count}/{len(input_files)}] 已处理: {filename}")
                    
                    datasets.append(ds)
            except Exception as e:
                print(f"❌ 处理失败: {filename} - {str(e)}")
                continue

        # 统计信息
        print(f"\n📊 处理统计:")
        print(f"   新处理: {processed_count} 个文件")
        print(f"   已跳过: {skipped_count} 个文件")
        print(f"   总计: {len(datasets)} 个文件用于合并")

        # 每个子文件夹单独合并
        if datasets:
            try:
                if has_level:
                    # 3D数据：沿level维度合并
                    ds_all = xr.concat(datasets, dim="level")
                else:
                    # 2D数据：直接合并（沿时间或其他维度）
                    ds_all = xr.concat(datasets, dim="time") if "time" in datasets[0].dims else xr.merge(datasets)
                
                var_data = ds_all[new_var_name]
                var_data.to_netcdf(merged_file)
                print(f"🎉 {folder_name} 合并完成!")
                print(f"   保存到: {merged_file}")
                print(f"   形状: {var_data.shape}")
                print(f"   维度: {list(var_data.dims)}")
            except Exception as e:
                print(f"❌ 合并失败: {str(e)}")
        else:
            print(f"⚠️ 文件夹 {folder_name} 没有可处理的文件！")


## merge_files

In [8]:
# input_folders1 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/ua_p4k_layers",

# ]

# input_folders2 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/va_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/va_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/va_p4k_layers"
# ]

# input_folders3 = [
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_4co2_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_cntl_layers",
#     "/work/mh1498/m301257/processed_data/3d_layers/pfull_p4k_layers"
# ]
# base_output_folder = "/work/mh1498/m301257"

# # 处理3D数据 (hus - 比湿)
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders1,
#     base_out_path=base_output_folder,
#     pattern="ua_lev_*.nc",
#     old_var_name="ua",
#     new_var_name="ua",
#     has_level=True,  # 3D数据，有level维度,
#     merged_file_name="ua_all_levels.nc"
# )

# # 处理3D数据 (ta - 温度)
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders2,
#     base_out_path=base_output_folder,
#     pattern="va_lev_*.nc",
#     old_var_name="va",
#     new_var_name="va",
#     has_level=True , # 3D数据，有level维度
#     merged_file_name="va_all_levels.nc"
# )


# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders3,
#     base_out_path=base_output_folder,
#     pattern="pfull_lev_*.nc",
#     old_var_name="pfull",
#     new_var_name="pfull",
#     has_level=True , # 3D数据，有level维度
#     merged_file_name="pfull_all_levels.nc"
# )

## 📝 验证 Level 维度自动检测

测试不同变量的 level 维度：
- `ua`, `va`, `phalf` → 使用 `level_half`
- `wa`, `hus`, `ta`, `pfull` → 使用 `level_full`
- 其他2D变量 → 无 level 维度

In [9]:
# 测试自动检测 level 维度功能
def test_level_detection(var_names):
    """测试不同变量的 level 维度检测"""
    print("="*70)
    print("🔍 测试变量 Level 维度自动检测")
    print("="*70)
    
    for var_name in var_names:
        try:
            # 加载变量
            var_temp = cat.ICON.C5.AMIP_CNTL.to_dask()[var_name].sel(time=slice("1980", "1981"))
            
            # 检测 level 维度
            possible_level_dims = ['level_full', 'level_half', 'level']
            level_dim_name = None
            
            for dim in possible_level_dims:
                if dim in var_temp.dims:
                    level_dim_name = dim
                    break
            
            if level_dim_name:
                n_levels = len(var_temp[level_dim_name])
                print(f"✅ {var_name:10s} → {level_dim_name:15s} (共 {n_levels} 层)")
            else:
                print(f"ℹ️  {var_name:10s} → 无 level 维度 (2D变量)")
                
        except Exception as e:
            print(f"❌ {var_name:10s} → 错误: {str(e)}")
    
    print("="*70)

# 测试变量列表
test_vars = ['ua', 'va', 'phalf', 'pfull', 'wa', 'hus', 'ta', 'pr', 'hfls', 'ts']
test_level_detection(test_vars)

🔍 测试变量 Level 维度自动检测
✅ ua         → level_half      (共 26 层)
✅ va         → level_half      (共 26 层)
✅ phalf      → level_half      (共 26 层)
✅ pfull      → level_full      (共 26 层)
✅ wa         → level_full      (共 26 层)
✅ hus        → level_full      (共 26 层)
✅ ta         → level_full      (共 26 层)
ℹ️  pr         → 无 level 维度 (2D变量)
ℹ️  hfls       → 无 level 维度 (2D变量)
ℹ️  ts         → 无 level 维度 (2D变量)


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),
